<a href="https://colab.research.google.com/github/Bhavika-G-Patil/learning-genai/blob/main/day3_embeddings_vectordb_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install sentence-transformers chromadb

In [12]:
from sentence_transformers import SentenceTransformer

# Load a pretrained embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Some sample sentences
sentences = [
    "The cat sat on the mat",
    "A dog lay on the rug",
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "The weather is sunny today"
]

# Convert sentences to embeddings
embeddings = model.encode(sentences)

print("Number of sentences:", len(sentences))
print("Embedding shape:", embeddings.shape)
print("\nFirst embedding (first 10 numbers):", embeddings[0][:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of sentences: 5
Embedding shape: (5, 384)

First embedding (first 10 numbers): [ 0.13040185 -0.01187011 -0.02811703  0.05123864 -0.05597443  0.03019155
  0.03016133  0.02469843 -0.01837055  0.05876681]


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute similarity between all sentence pairs
similarity_matrix = cosine_similarity(embeddings)

print("Similarity Matrix:")
print("(rows and columns = our 5 sentences)\n")

for i, s1 in enumerate(sentences):
    for j, s2 in enumerate(sentences):
        if j > i:  # avoid duplicates
            score = similarity_matrix[i][j]
            print(f"'{s1[:30]}...' vs '{s2[:30]}...'")
            print(f"Similarity: {score:.4f}\n")

Similarity Matrix:
(rows and columns = our 5 sentences)

'The cat sat on the mat...' vs 'A dog lay on the rug...'
Similarity: 0.4913

'The cat sat on the mat...' vs 'Machine learning is a subset o...'
Similarity: -0.0473

'The cat sat on the mat...' vs 'Deep learning uses neural netw...'
Similarity: -0.0750

'The cat sat on the mat...' vs 'The weather is sunny today...'
Similarity: 0.0116

'A dog lay on the rug...' vs 'Machine learning is a subset o...'
Similarity: -0.0035

'A dog lay on the rug...' vs 'Deep learning uses neural netw...'
Similarity: -0.0415

'A dog lay on the rug...' vs 'The weather is sunny today...'
Similarity: 0.0230

'Machine learning is a subset o...' vs 'Deep learning uses neural netw...'
Similarity: 0.4251

'Machine learning is a subset o...' vs 'The weather is sunny today...'
Similarity: 0.0151

'Deep learning uses neural netw...' vs 'The weather is sunny today...'
Similarity: -0.0420



In [16]:
import chromadb

# Create a local Chroma client (runs in memory)
client = chromadb.Client()

# Create a collection - like a table in a regular database
# but stores vectors instead of just text
collection = client.create_collection(
    name="my_second_vectordb",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for search
)

print("ChromaDB collection created successfully!")
print("Collection name:", collection.name)

ChromaDB collection created successfully!
Collection name: my_second_vectordb


In [17]:
# Add sentences and their embeddings to the collection
collection.add(
    documents=sentences,           # original text
    embeddings=embeddings.tolist(), # embedding vectors
    ids=[f"doc_{i}" for i in range(len(sentences))]  # unique ID for each
)

print(f"Stored {collection.count()} documents in ChromaDB")

Stored 5 documents in ChromaDB


In [18]:
# Query with a brand new sentence never seen before
query_text = "I love my pet animals"

# Convert query to embedding
query_embedding = model.encode([query_text]).tolist()

# Search ChromaDB for 2 most similar sentences
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2  # return top 2 most similar
)

print("Query:", query_text)
print("\nTop 2 most similar sentences found:")
for i, doc in enumerate(results['documents'][0]):
    distance = results['distances'][0][i]
    print(f"\n{i+1}. '{doc}'")
    print(f"   Similarity score: {1-distance:.4f}")

Query: I love my pet animals

Top 2 most similar sentences found:

1. 'A dog lay on the rug'
   Similarity score: 0.2060

2. 'The cat sat on the mat'
   Similarity score: 0.2058
